# Script 4 — Avaliação Aprofundada, Análise de Risco e Estresse Probabilístico

**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

Este script consome os artefatos do `03_cvm_modelagem.ipynb` e realiza avaliação
completa sobre **todos os 36 targets treinados** (9 variáveis × 4 horizontes),
além das análises de risco corporativo.

| Etapa | Conteúdo |
|-------|----------|
| 0 | Imports, logging, constantes espelhadas do Script 3 |
| 1 | Carga de todos os artefatos do Script 3 |
| 2 | Funções de métricas com inversão de transformações (escala original R$ mil) |
| 3 | Relatório consolidado — todos os 36 targets × 4 algoritmos |
| 4 | Diagnóstico CV vs. Teste (detecção de overfitting por target) |
| 5 | Análise estratificada por setor — todos os 36 targets |
| 6 | Z''-Score de Altman histórico e prospectivo |
| 7 | Score de Risco Composto (0–100, 8 gatilhos) |
| 8 | Análise de Estresse Monte Carlo (500 sim., σ=15%) |
| 8b | Conformal Prediction — IC 90% com garantia estatística |
| 9 | Feature Importance agregada com ranking por família |
| 10 | Análise de resíduos + teste de viés sistemático |
| 11 | Visualizações (6 figuras) |
| 12 | Persistência de todos os artefatos para o Script 5 |

---
**Referências:**
- Z''-Score: Altman (1968, 1995) | SMAPE: Hyndman & Athanasopoulos (2018)
- RobustScaler / inversão de transformações: Pedregosa et al. (2011)
- Análise de estresse σ=15%: Damodaran (2012)

## Etapa 0. Imports, Configuração e Logging

In [1]:
import json
import logging
import pickle
import warnings
from collections import Counter
from datetime import datetime
from pathlib import Path

import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats as sp_stats

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 200)

PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)
(PASTA_SAIDA / 'logs').mkdir(exist_ok=True)
(PASTA_SAIDA / 'figuras').mkdir(exist_ok=True)

# ── Logging ───────────────────────────────────────────────────────────────────
logger = logging.getLogger('pipeline_avaliacao_v4')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s',
                         datefmt='%Y-%m-%d %H:%M:%S')
_sh = logging.StreamHandler(); _sh.setLevel(logging.INFO); _sh.setFormatter(_fmt)
logger.addHandler(_sh)
_fh = logging.FileHandler(PASTA_SAIDA / 'logs' / 'pipeline_avaliacao.log',
                           mode='w', encoding='utf-8')
_fh.setLevel(logging.DEBUG); _fh.setFormatter(_fmt)
logger.addHandler(_fh)

# ── Constantes IDÊNTICAS ao Script 3 ─────────────────────────────────────────
# Devem ser mantidas em sincronia com 03_cvm_modelagem.ipynb
_TARGET_BASES = [
    'DRE_3.01', 'DRE_3.11', 'EBITDA',
    'BPA_1', 'BPA_1.01', 'BPP_2.01', 'BPP_2.03', 'BPP_2',
    'DFC_MI_6.01',
]
_HORIZONTES = ['_ITR_T1', '_ITR_T2', '_ITR_T3', '_DFP']

# 9 variáveis × 4 horizontes = 36 targets treinados
TODOS_TARGETS = [f'TARGET_{b}{h}' for b in _TARGET_BASES for h in _HORIZONTES]

# Transformações por base (log1p → targets positivos | arcsinh → podem ser negativos)
_LOG_BASES     = {'DRE_3.01','EBITDA','BPA_1','BPA_1.01','BPP_2.01','BPP_2.03','BPP_2'}
_ARCSINH_BASES = {'DFC_MI_6.01','DRE_3.11'}
LOG_TARGETS     = {f'TARGET_{b}{h}' for b in _LOG_BASES     for h in _HORIZONTES}
ARCSINH_TARGETS = {f'TARGET_{b}{h}' for b in _ARCSINH_BASES for h in _HORIZONTES}

# Nomes legíveis para relatórios
NOME_VARIAVEL = {
    'DRE_3.01':    'Receita Líquida',
    'DRE_3.11':    'Lucro Líquido',
    'EBITDA':      'EBITDA',
    'BPA_1':       'Ativo Total',
    'BPA_1.01':    'Ativo Circulante',
    'BPP_2.01':    'Passivo Circulante',
    'BPP_2.03':    'Patrimônio Líquido',
    'BPP_2':       'Passivo Total',
    'DFC_MI_6.01': 'FCO',
}

# Foco primário do TCC (Receita, Lucro, EBITDA) — usados em análises detalhadas
BASES_FOCO  = ['DRE_3.01','DRE_3.11','EBITDA']
TARGETS_FOCO = [f'TARGET_{b}{h}' for b in BASES_FOCO for h in _HORIZONTES]  # 12 targets

# Parâmetros de risco e estresse
ZONA_SEGURA    = 2.60
ZONA_CINZA_INF = 1.10
N_SIM_MC       = 500
SIGMA_MC       = 0.15
SEED_MC        = 42
COVID_ANOS     = {2020, 2021}

logger.info('Script 4 iniciado em %s | %d targets totais | %d foco TCC',
            datetime.now().strftime('%Y-%m-%d %H:%M:%S'), len(TODOS_TARGETS), len(TARGETS_FOCO))
print(f'✅ Etapa 0 — {len(TODOS_TARGETS)} targets configurados ({len(TARGETS_FOCO)} foco TCC)')

2026-06-01 14:30:30 | INFO     | Script 4 iniciado em 2026-06-01 14:30:30 | 36 targets totais | 12 foco TCC


✅ Etapa 0 — 36 targets configurados (12 foco TCC)


## Etapa 1. Carga dos Artefatos do Script 3

In [2]:
def _pkl(nome, req=True):
    p = PASTA_SAIDA / nome
    if not p.exists():
        msg = f'{nome} não encontrado — execute 03_cvm_modelagem.ipynb antes.'
        if req: raise FileNotFoundError(msg)
        logger.warning(msg); return None
    with open(p,'rb') as f: obj = pickle.load(f)
    logger.info('Carregado: %s', nome)
    return obj

# Artefatos obrigatórios
treino  = pd.read_parquet(PASTA_SAIDA / 'treino.parquet')
teste   = pd.read_parquet(PASTA_SAIDA / 'teste.parquet')
melhores                     = _pkl('melhores_modelos.pkl')
metricas_teste_pkl           = _pkl('metricas_teste.pkl')
baselines                    = _pkl('baselines.pkl')
feature_importances          = _pkl('feature_importances.pkl')
selected_features_por_target = _pkl('selected_features_por_target.pkl')
features_por_target_alg      = _pkl('features_por_target_alg.pkl', req=False) or {}

# Opcionais (enriquecem análises quando disponíveis)
FEATURES          = _pkl('features.pkl',          req=False) or []
KPIS              = _pkl('kpis.pkl',              req=False) or []
TARGETS_POR_HOR   = _pkl('targets_por_horizonte.pkl', req=False) or {}
TARGET_COLS_SOURCE= _pkl('target_cols_source.pkl',req=False) or {}

# CSVs de resultado do Script 3
df_cv   = pd.read_csv(PASTA_SAIDA/'resultados_cv.csv')   if (PASTA_SAIDA/'resultados_cv.csv').exists()   else pd.DataFrame()
df_te   = pd.read_csv(PASTA_SAIDA/'resultados_teste.csv') if (PASTA_SAIDA/'resultados_teste.csv').exists() else pd.DataFrame()

# Predições detalhadas linha a linha (geradas na Etapa 6 do Script 3)
_pp = PASTA_SAIDA/'predicoes_teste_detalhadas.parquet'
df_pred = pd.read_parquet(_pp) if _pp.exists() else pd.DataFrame()

# Predições prospectivas 2026 (geradas na Etapa Prospectiva do Script 3)
_pp2 = PASTA_SAIDA/'predicoes_prospectivas.parquet'
df_prosp = pd.read_parquet(_pp2) if _pp2.exists() else pd.DataFrame()

# TARGETS ativos: apenas os que foram de fato treinados (presentes em melhores)
TARGETS = [t for t in TODOS_TARGETS if t in melhores]

# Dataset consolidado para Z-Score (usa parquet se disponível, senão concatena splits)
_pds = PASTA_SAIDA/'dataset_cvm_consolidado.parquet'
dataset = pd.read_parquet(_pds) if _pds.exists() else pd.concat([treino,teste],ignore_index=True)

# Normaliza datas (remove timezone para evitar erros em joins)
for _df in [treino, teste, dataset]:
    for _c in ('DT_REFER','DT_TARGET','DT_TARGET_DFP'):
        if _c in _df.columns:
            _df[_c] = pd.to_datetime(_df[_c], utc=True, errors='coerce').dt.tz_localize(None)

# Mapas de lookup CNPJ → metadados
mapa_setor = {}; mapa_nome = {}
if 'CNPJ_CIA' in dataset.columns:
    _b = dataset.drop_duplicates('CNPJ_CIA')
    if 'SETOR'   in _b.columns: mapa_setor = _b.set_index('CNPJ_CIA')['SETOR'].to_dict()
    if 'NOME_CIA'in _b.columns: mapa_nome  = _b.set_index('CNPJ_CIA')['NOME_CIA'].to_dict()

print(f'Treino   : {len(treino):,} obs | {treino.shape[1]} colunas')
print(f'Teste    : {len(teste):,} obs  | {teste.shape[1]} colunas')
print(f'Dataset  : {len(dataset):,} obs | {dataset.shape[1]} colunas')
print(f'Targets treinados : {len(TARGETS)} / {len(TODOS_TARGETS)} possíveis')
print(f'Predições detalhadas : {len(df_pred):,} linhas')
print(f'Predições prospectivas : {len(df_prosp):,} linhas')
print(f'Algoritmos : {sorted(set(melhores.values()))}')
if mapa_setor:
    print(f'Setores  : {sorted(set(mapa_setor.values()))}')
logger.info('Artefatos OK | targets=%d | treino=%d | teste=%d', len(TARGETS), len(treino), len(teste))

2026-06-01 14:30:39 | INFO     | Carregado: melhores_modelos.pkl
2026-06-01 14:30:39 | INFO     | Carregado: metricas_teste.pkl
2026-06-01 14:30:39 | INFO     | Carregado: baselines.pkl
2026-06-01 14:30:39 | INFO     | Carregado: feature_importances.pkl
2026-06-01 14:30:39 | INFO     | Carregado: selected_features_por_target.pkl
2026-06-01 14:30:39 | WARNING  | features_por_target_alg.pkl não encontrado — execute 03_cvm_modelagem.ipynb antes.
2026-06-01 14:30:39 | INFO     | Carregado: features.pkl
2026-06-01 14:30:39 | INFO     | Carregado: kpis.pkl
2026-06-01 14:30:39 | INFO     | Carregado: targets_por_horizonte.pkl
2026-06-01 14:30:40 | INFO     | Carregado: target_cols_source.pkl
2026-06-01 14:30:40 | INFO     | Artefatos OK | targets=36 | treino=813 | teste=193


Treino   : 813 obs | 991 colunas
Teste    : 193 obs  | 991 colunas
Dataset  : 1,031 obs | 775 colunas
Targets treinados : 36 / 36 possíveis
Predições detalhadas : 28,035 linhas
Predições prospectivas : 828 linhas
Algoritmos : ['GradientBoosting', 'RandomForest']
Setores  : ['Commodities', 'Energia', 'Petróleo', 'Tecnologia', 'Varejo']


## Etapa 2. Funções de Métricas com Inversão de Transformações

Todas as métricas são calculadas na **escala original em R$ mil**, após inversão
das transformações aplicadas no Script 3 (`expm1` para log1p; `sinh` para arcsinh).
Isso garante interpretabilidade financeira direta (Hyndman & Athanasopoulos, 2018).

In [4]:
def get_transform(target):
    if target in LOG_TARGETS:     return 'log1p'
    if target in ARCSINH_TARGETS: return 'arcsinh'
    return 'none'

def inv_transform(y, t='none'):
    y = np.asarray(y, float)
    if t == 'log1p':   return np.expm1(y)
    if t == 'arcsinh': return np.sinh(y)
    return y

def _smape(yt, yp):
    yt, yp = np.asarray(yt,float), np.asarray(yp,float)
    d = (np.abs(yt)+np.abs(yp))/2.0
    m = d > 1e-9
    return float(np.mean(np.abs(yt[m]-yp[m])/d[m])) if m.sum()>0 else np.nan

def _mape(yt, yp):
    yt, yp = np.asarray(yt,float), np.asarray(yp,float)
    m = np.abs(yt) > 1e-9
    return float(np.mean(np.abs((yt[m]-yp[m])/yt[m]))) if m.sum()>0 else np.nan

def _rmse(yt, yp):
    return float(np.sqrt(np.mean((np.asarray(yt,float)-np.asarray(yp,float))**2)))

def _mae(yt, yp):
    return float(np.mean(np.abs(np.asarray(yt,float)-np.asarray(yp,float))))

def _r2(yt, yp):
    yt, yp = np.asarray(yt,float), np.asarray(yp,float)
    if len(yt)<2 or np.isclose(np.var(yt),0): return np.nan
    ss_r = np.sum((yt-yp)**2); ss_t = np.sum((yt-yt.mean())**2)
    return float(1-ss_r/ss_t) if ss_t>0 else np.nan

def _theil(yt, yp):
    yt, yp = np.asarray(yt,float), np.asarray(yp,float)
    if len(yt)<2: return np.nan
    em = np.sqrt(np.mean((yt[1:]-yp[1:])**2))
    en = np.sqrt(np.mean((yt[1:]-yt[:-1])**2))
    return float(em/en) if en>1e-9 else np.nan

def _da(yt, yp):
    yt, yp = np.asarray(yt,float), np.asarray(yp,float)
    if len(yt)<2: return np.nan
    return float(np.mean(np.sign(yt[1:]-yt[:-1])==np.sign(yp[1:]-yp[:-1])))

def metricas(yt, yp, target=None):
    """Retorna todas as métricas. Se target fornecido, inverte transformação antes."""    
    if target:
        t = get_transform(target)
        yt = inv_transform(yt, t); yp = inv_transform(yp, t)
    return {'SMAPE':_smape(yt,yp),'MAPE':_mape(yt,yp),'RMSE':_rmse(yt,yp),
            'MAE':_mae(yt,yp),'R2':_r2(yt,yp),'TheilU':_theil(yt,yp),
            'DA':_da(yt,yp),'N':int(len(yt))}

def metricas_macro(df, target, yt_col='y_true', yp_col='y_pred', grp='CNPJ_CIA'):
    """Média aritmética das métricas por empresa — evita dominância de grandes companhias."""    
    rows=[]
    for _, g in df.groupby(grp):
        yt=g[yt_col].values; yp=g[yp_col].values
        mask=np.isfinite(yt)&np.isfinite(yp)
        if mask.sum()<2: continue
        rows.append(metricas(yt[mask], yp[mask], target))
    if not rows: return {}
    df_m=pd.DataFrame(rows)
    return {f'{k}_macro':float(df_m[k].mean()) for k in df_m.columns if k!='N'}

print('✅ Etapa 2 — funções de métricas prontas')

✅ Etapa 2 — funções de métricas prontas


## Etapa 3. Relatório Consolidado — Todos os 36 Targets × 4 Algoritmos

Consolida as métricas já calculadas pelo Script 3 e adiciona colunas derivadas:
cobertura de baseline, ranking de algoritmo por target e sinalização de overfitting.

In [5]:
print('='*100)
print(f'DESEMPENHO NO HOLD-OUT 2024–2025 — TODOS OS {len(TARGETS)} TARGETS TREINADOS')
print('='*100)

_mc_cv  = 'SMAPE_CV_macro_empresa'
_mc_te  = 'SMAPE_teste_macro_empresa'
_mc_theil = 'TheilU_teste_macro_empresa'
_mc_da  = 'DA_teste_macro_empresa'
_mc_r2  = 'R2_teste_macro_empresa'

if not df_te.empty:
    # ── 3.1 Tabela completa por horizonte × algoritmo ─────────────────────────
    cols_disp = [c for c in [_mc_te,'RMSE_teste_macro_empresa',_mc_r2,_mc_theil,_mc_da]
                 if c in df_te.columns]

    for horizonte in _HORIZONTES:
        df_h = df_te[df_te['Horizonte']==horizonte] if 'Horizonte' in df_te.columns else df_te
        if df_h.empty: continue
        print(f'\n--- Horizonte: {horizonte.replace("_","")} ---')
        if cols_disp:
            g = df_h.groupby('Algoritmo')[cols_disp].mean()
            print(g.round(4).to_string())

    # ── 3.2 Contagem de vitórias (todos os 36 targets) ────────────────────────
    print('\n' + '='*70)
    print(f'RANKING DE ALGORITMOS — {len(melhores)} targets (critério: SMAPE_CV macro)')
    print('='*70)
    cnt = Counter(melhores.values())
    for alg, n in sorted(cnt.items(), key=lambda x:-x[1]):
        pct = n/len(melhores)*100
        print(f'  {alg:<22} {n:>3}× ({pct:5.1f}%) {"█"*int(pct/3)}')

    # ── 3.3 Contagem por grupo de variável ────────────────────────────────────
    print('\n--- Vitórias por grupo de variável ---')
    for base in _TARGET_BASES:
        vits = {h: melhores.get(f'TARGET_{base}{h}','?') for h in _HORIZONTES}
        linha = '  '.join(f'{h.replace("_","")}:{v}' for h,v in vits.items())
        print(f'  {NOME_VARIAVEL.get(base,base):<22} | {linha}')

    # ── 3.4 Tabela completa exportável ───────────────────────────────────────
    df_te['base_variavel'] = df_te['Target'].apply(
        lambda t: t.replace('TARGET_','').rsplit('_ITR',1)[0].rsplit('_DFP',1)[0]
    )
    df_te['nome_variavel'] = df_te['base_variavel'].map(NOME_VARIAVEL).fillna(df_te['base_variavel'])
    df_te['melhor_target'] = df_te.apply(
        lambda r: melhores.get(r['Target'])==r['Algoritmo'], axis=1
    )
    df_te.to_csv(PASTA_SAIDA/'resultados_teste_enriquecido.csv', index=False)
    print(f'\n✅ resultados_teste_enriquecido.csv salvo ({len(df_te)} linhas)')
else:
    print('⚠️  resultados_teste.csv não encontrado.')

print('\n✅ Etapa 3 concluída')

DESEMPENHO NO HOLD-OUT 2024–2025 — TODOS OS 36 TARGETS TREINADOS

--- Horizonte: ITRT1 ---
                  SMAPE_teste_macro_empresa  RMSE_teste_macro_empresa  R2_teste_macro_empresa  TheilU_teste_macro_empresa  DA_teste_macro_empresa
Algoritmo                                                                                                                                        
Ensemble                             0.2654            1,628,257.2792                 -0.4101                      0.9537                  0.5079
GradientBoosting                     0.2670            1,671,545.6902                 -0.3719                      1.0641                  0.5238
RandomForest                         0.2868            2,018,028.9292                 -0.8100                      1.1579                  0.4603
Ridge                                0.5038            5,706,788.6007                -10.9255                      1.5943                  0.3968
SVR                              

## Etapa 4. Diagnóstico CV vs. Teste — Detecção de Overfitting por Target

In [6]:
print('='*90)
print('DIAGNÓSTICO OVERFITTING — SMAPE_CV vs SMAPE_Teste (todos os targets)')
print('Δ > 0.10 → possível overfitting  |  Δ < 0 → modelo generaliza melhor que no CV')
print('='*90)

if not df_cv.empty and not df_te.empty:
    _cv_col  = 'SMAPE_CV_macro_empresa'
    _te_col  = 'SMAPE_teste_macro_empresa'
    _u_col   = 'TheilU_teste_macro_empresa'

    rows_diag = []
    for target in TARGETS:
        alg = melhores.get(target)
        if not alg: continue
        cv_row = df_cv[(df_cv['Target']==target)&(df_cv['Algoritmo']==alg)]
        te_row = df_te[(df_te['Target']==target)&(df_te['Algoritmo']==alg)]
        if cv_row.empty or te_row.empty: continue

        s_cv = float(cv_row[_cv_col].iloc[0]) if _cv_col in cv_row else np.nan
        s_te = float(te_row[_te_col].iloc[0]) if _te_col in te_row else np.nan
        u    = float(te_row[_u_col].iloc[0])  if _u_col  in te_row else np.nan
        delta = s_te - s_cv if pd.notna(s_cv) and pd.notna(s_te) else np.nan
        base  = target.replace('TARGET_','').rsplit('_ITR',1)[0].rsplit('_DFP',1)[0]
        hor   = next((h for h in _HORIZONTES if target.endswith(h)),'')
        rows_diag.append({
            'Target':target,'Variavel':NOME_VARIAVEL.get(base,base),
            'Horizonte':hor,'Algoritmo':alg,
            'SMAPE_CV':s_cv,'SMAPE_Teste':s_te,'Delta':delta,
            'TheilU':u,'U_ok':u<1 if pd.notna(u) else None,
        })

    df_diag = pd.DataFrame(rows_diag)
    if not df_diag.empty:
        # Exibe agrupado por variável
        for base in _TARGET_BASES:
            sub = df_diag[df_diag['Variavel']==NOME_VARIAVEL.get(base,base)]
            if sub.empty: continue
            print(f'\n{NOME_VARIAVEL.get(base,base)}:')
            for _, r in sub.iterrows():
                flag = ' ⚠️ OVERFIT' if (pd.notna(r.Delta) and r.Delta>0.10) else ''
                u_flag = f' U={r.TheilU:.3f}' if pd.notna(r.TheilU) else ''
                beat = '✅' if r.get('U_ok') else '❌'
                print(f'  {r.Horizonte.replace("_",""):<8} CV={r.SMAPE_CV:.3f}  '
                      f'Teste={r.SMAPE_Teste:.3f}  Δ={r.Delta:+.3f}{u_flag} '
                      f'{beat} baseline{flag}')
        df_diag.to_csv(PASTA_SAIDA/'diagnostico_overfitting.csv', index=False)
        print(f'\n✅ diagnostico_overfitting.csv salvo')
else:
    print('⚠️  CSVs de CV ou teste não disponíveis.')

print('\n✅ Etapa 4 concluída')

DIAGNÓSTICO OVERFITTING — SMAPE_CV vs SMAPE_Teste (todos os targets)
Δ > 0.10 → possível overfitting  |  Δ < 0 → modelo generaliza melhor que no CV

Receita Líquida:
  ITRT1    CV=0.165  Teste=0.111  Δ=-0.054 U=0.152 ✅ baseline
  ITRT2    CV=0.150  Teste=0.100  Δ=-0.050 U=0.179 ✅ baseline
  ITRT3    CV=0.144  Teste=0.119  Δ=-0.024 U=0.214 ✅ baseline
  DFP      CV=0.152  Teste=0.194  Δ=+0.042 U=2.000 ❌ baseline

Lucro Líquido:
  ITRT1    CV=0.746  Teste=0.608  Δ=-0.138 U=0.571 ✅ baseline
  ITRT2    CV=0.717  Teste=0.775  Δ=+0.058 U=0.770 ✅ baseline
  ITRT3    CV=0.840  Teste=0.764  Δ=-0.076 U=0.778 ✅ baseline
  DFP      CV=0.533  Teste=0.469  Δ=-0.064 U=1.619 ❌ baseline

EBITDA:
  ITRT1    CV=0.154  Teste=0.282  Δ=+0.128 U=0.330 ✅ baseline ⚠️ OVERFIT
  ITRT2    CV=0.197  Teste=0.155  Δ=-0.042 U=0.254 ✅ baseline
  ITRT3    CV=0.145  Teste=0.087  Δ=-0.058 U=0.196 ✅ baseline
  DFP      CV=0.135  Teste=0.102  Δ=-0.033 U=2.000 ❌ baseline

Ativo Total:
  ITRT1    CV=0.099  Teste=0.077  Δ=-0.0

## Etapa 5. Análise por Setor — Todos os 36 Targets

In [ ]:
rows_setor = []

if not df_pred.empty and mapa_setor:
    df_ps = df_pred.copy()
    df_ps['SETOR']    = df_ps['CNPJ_CIA'].map(mapa_setor)
    df_ps['NOME_CIA'] = df_ps['CNPJ_CIA'].map(mapa_nome)

    df_best = df_ps[df_ps.apply(
        lambda r: melhores.get(r['Target']) == r['Algoritmo'], axis=1
    )].copy()

    for (setor, target), grp in df_best.groupby(['SETOR', 'Target']):
        yt_raw = grp['y_true'].values
        yp_raw = grp['y_pred'].values
        mask   = np.isfinite(yt_raw) & np.isfinite(yp_raw)
        if mask.sum() < 2: continue

        transf = get_transform(target)
        # Script 3 salva y_true em escala TRANSFORMADA e y_pred em escala ORIGINAL (R$).
        # Aplica inv_transform apenas em y_true. y_pred ja esta em R$.
        yt_i = inv_transform(yt_raw[mask], transf)
        yp_i = yp_raw[mask].astype(float)

        mask2 = np.isfinite(yt_i) & np.isfinite(yp_i)
        if mask2.sum() < 2: continue

        m    = metricas(yt_i[mask2], yp_i[mask2])
        base = target.replace('TARGET_', '').rsplit('_ITR', 1)[0].rsplit('_DFP', 1)[0]
        hor  = next((h for h in _HORIZONTES if target.endswith(h)), 'N/A')
        rows_setor.append({
            'SETOR': setor, 'Target': target,
            'Variavel': NOME_VARIAVEL.get(base, base), 'Horizonte': hor,
            'Algoritmo': melhores.get(target, '?'), **m
        })

if rows_setor:
    df_setor = pd.DataFrame(rows_setor)
    df_setor.to_csv(PASTA_SAIDA / 'metricas_por_setor.csv', index=False)

    print('=== SMAPE por Setor x Variavel x Horizonte ===')
    pv = df_setor.pivot_table(values='SMAPE', index='Variavel',
                               columns='SETOR', aggfunc='mean')
    print(pv.round(3).to_string())

    print('\n=== U de Theil medio por Setor (< 1 = supera naive) ===')
    print(df_setor.groupby('SETOR')['TheilU'].agg(['mean','median']).round(3).to_string())

    print('\n=== Ranking de previsibilidade por setor (SMAPE medio) ===')
    rank = df_setor.groupby('SETOR')['SMAPE'].mean().sort_values()
    for s, v in rank.items():
        bar = chr(9608) * int((1 - min(v, 1)) * 20)
        print(f'  {s:<18} SMAPE={v:.3f}  {bar}')
    logger.info('Analise setor: %d combinacoes', len(df_setor))
else:
    df_setor = pd.DataFrame()
    print('AVISO: predicoes_teste_detalhadas.parquet ou mapa_setor indisponivel.')

print('\nOK Etapa 5 concluida')


## Etapa 6. Z''-Score de Altman para Mercados Emergentes

$$Z'' = 6{,}56\,X_1 + 3{,}26\,X_2 + 6{,}72\,X_3 + 1{,}05\,X_4$$

Aplicado sobre KPIs **históricos** (diagnóstico) e sobre os KPIs
**preditos pelos melhores modelos** (classificação prospectiva).
Zonas: Z'' > 2,60 → Segura | 1,10–2,60 → Cinza | < 1,10 → Insolvência.

In [ ]:
def classificar_zona(z):
    if pd.isna(z):           return 'N/D'
    if z > ZONA_SEGURA:      return 'Segura'
    if z >= ZONA_CINZA_INF:  return 'Cinza'
    return 'Insolvencia'

def calcular_zpp(row, mapa_cols):
    def g(k):
        c = mapa_cols.get(k)
        if not c: return np.nan
        v = row.get(c, np.nan)
        return float(v) if pd.notna(v) else np.nan

    ac     = g('ac')
    pc     = g('pc')
    at     = g('at')
    pl     = g('pl')
    ebit   = g('ebit')
    ebitda = g('ebitda')
    pt     = g('pt')   # Passivo Total (BPP_2) — usado em X4 e pass_total

    if np.isnan(ebit) and not np.isnan(ebitda):
        ebit = ebitda * 0.85

    # Usa Passivo Total (BPP_2) diretamente como denominador de X4.
    # Versao anterior calculava pass_total = pc + pnc onde pnc estava mapeado
    # erroneamente para BPP_2.03 (Patrimonio Liquido), tornando pass_total = PC + PL.
    pass_total = pt

    X1 = ((ac - pc) / at
          if (not np.isnan(at) and at > 0 and not np.isnan(ac) and not np.isnan(pc))
          else np.nan)
    X2 = (pl / at
          if (not np.isnan(at) and at > 0 and not np.isnan(pl))
          else np.nan)
    X3 = (ebit / at
          if (not np.isnan(at) and at > 0 and not np.isnan(ebit))
          else np.nan)
    X4 = (pl / pass_total
          if (not np.isnan(pass_total) and pass_total > 0 and not np.isnan(pl))
          else np.nan)

    n_ok = sum(not np.isnan(v) for v in [X1, X2, X3, X4])
    if n_ok < 3:
        return np.nan, np.nan, np.nan, np.nan, np.nan, n_ok

    # Correcao: `X or 0` nao funciona com np.nan porque nan e truthy em Python.
    # np.nan or 0 retorna np.nan, nao 0. Usa funcao explicita.
    def _v(x): return 0.0 if np.isnan(x) else x

    z = 6.56 * _v(X1) + 3.26 * _v(X2) + 6.72 * _v(X3) + 1.05 * _v(X4)
    return z, X1, X2, X3, X4, n_ok

def _mapa_cols(df):
    def _p(*cs): return next((c for c in cs if c in df.columns), None)
    return {
        'ac':     _p('BPA_1.01', 'ativo_circulante'),
        'pc':     _p('BPP_2.01', 'passivo_circulante'),
        'at':     _p('BPA_1',    'ativo_total'),
        # 'pl': apenas BPP_2.03 (Patrimonio Liquido).
        # Removido fallback BPP_2 que e Passivo Total — confundia X2 e X4.
        'pl':     _p('BPP_2.03', 'patrimonio_liquido'),
        # 'pt': Passivo Total (BPP_2). Substitui o calculo pc + pnc onde
        # pnc estava incorretamente mapeado para BPP_2.03 (Patrimonio Liquido).
        'pt':     _p('BPP_2', 'passivo_total'),
        'ebit':   _p('EBIT',  'ebit'),
        'ebitda': _p('EBITDA','ebitda'),
    }

if 'CNPJ_CIA' in dataset.columns:
    mapa = _mapa_cols(dataset)
    print(f"Colunas mapeadas para Z'': {mapa}")

    dfs_z = []
    for cnpj, grp in dataset.groupby('CNPJ_CIA'):
        grp_s = grp.sort_values('ANO') if 'ANO' in grp.columns else grp
        res   = []
        for _, row in grp_s.iterrows():
            z, X1, X2, X3, X4, nok = calcular_zpp(row, mapa)
            res.append({'altman_z_pp': z, 'zona_altman': classificar_zona(z),
                        'X1': X1, 'X2': X2, 'X3': X3, 'X4': X4, 'comp_ok': nok})
        meta = [c for c in ['CNPJ_CIA','NOME_CIA','ANO','SETOR','ORIGEM','DT_REFER']
                if c in grp.columns]
        dfs_z.append(
            grp_s[meta].reset_index(drop=True)
            .join(pd.DataFrame(res, index=grp_s.index).reset_index(drop=True))
        )

    df_zscore = pd.concat(dfs_z, ignore_index=True)
    df_zscore.to_csv(PASTA_SAIDA / 'altman_zscore.csv',      index=False)
    df_zscore.to_parquet(PASTA_SAIDA / 'altman_zscore.parquet', index=False)

    df_zv = df_zscore[df_zscore['altman_z_pp'].notna()]
    n_tot = len(df_zv)
    print(f"\n=== Z'' — {n_tot:,} observacoes validas ===")
    for zona, cnt in df_zv['zona_altman'].value_counts().items():
        print(f'  {zona:<15} {cnt:>5} ({cnt / n_tot * 100:5.1f}%)')
    if 'ANO' in df_zscore.columns:
        print("\nZ'' medio por ano:")
        print(df_zv.groupby('ANO')['altman_z_pp'].agg(['mean','median','std']).round(3).to_string())
    if 'SETOR' in df_zscore.columns:
        print("\nZ'' medio por setor:")
        print(df_zv.groupby('SETOR')['altman_z_pp']
              .agg(['mean','median','std','count']).round(3).to_string())
    logger.info("Z-Score: %d validos de %d", n_tot, len(df_zscore))
else:
    df_zscore = pd.DataFrame()
    print('AVISO: CNPJ_CIA ausente no dataset — Z-Score ignorado.')

print('\nOK Etapa 6 concluida')


## Etapa 7. Score de Risco Composto (0–100, 8 Gatilhos Financeiros)

In [ ]:
REGRAS_RISCO = [
    ('liquidez_corrente', 'abaixo', 1.0,  15, 'Liq. corrente < 1.0'),
    ('liquidez_imediata', 'abaixo', 0.3,  10, 'Liq. imediata < 0.3'),
    ('margem_liquida',    'abaixo', 0.0,  20, 'Margem líquida negativa'),
    ('roe',               'abaixo', 0.0,  10, 'ROE negativo'),
    ('endividamento',     'acima',  0.7,  15, 'Endividamento > 70%'),
    ('alavancagem_de',    'acima',  3.0,  10, 'D/E > 3×'),
    ('cobertura_juros',   'abaixo', 1.5,  15, 'Cobertura juros < 1.5×'),
    ('margem_ebitda',     'abaixo', 0.05,  5, 'Margem EBITDA < 5%'),
]

def score_risco(row):
    s=0
    for col,dir,lim,pts,_ in REGRAS_RISCO:
        if col not in row.index: continue
        v=row[col]
        if pd.isna(v): continue
        if (dir=='abaixo' and v<lim) or (dir=='acima' and v>lim): s+=pts
    return min(s,100)

def classe_risco(s):
    return 'Baixo' if s<20 else ('Moderado' if s<40 else ('Elevado' if s<60 else 'Crítico'))

kpis_disp = [r[0] for r in REGRAS_RISCO if r[0] in dataset.columns]
kpis_faltando = [r[0] for r in REGRAS_RISCO if r[0] not in dataset.columns]
print(f'KPIs de risco disponíveis : {kpis_disp}')
print(f'KPIs de risco ausentes    : {kpis_faltando}')

if kpis_disp:
    dataset['score_risco']  = dataset.apply(score_risco, axis=1)
    dataset['classe_risco'] = dataset['score_risco'].apply(classe_risco)
    print('\nDistribuição de classes de risco:')
    for cl, cnt in dataset['classe_risco'].value_counts().items():
        pct=cnt/len(dataset)*100
        print(f'  {cl:<10} {cnt:>6} ({pct:5.1f}%)')
    if 'SETOR' in dataset.columns:
        print('\nScore médio por setor:')
        print(dataset.groupby('SETOR')['score_risco']
              .agg(['mean','median','max']).round(1).to_string())
else:
    print('⚠️  Nenhum KPI de risco disponível no dataset.')

print('\n✅ Etapa 7 concluída')

## Etapa 8. Análise de Estresse Monte Carlo — Todas as Empresas

Para cada empresa do dataset e cada target DFP (horizonte anual), aplica
perturbação gaussiana σ=15% sobre o vetor de KPIs e gera N=500 predições.

**Destaque:** após processar todas, empresas representativas são sinalizadas
nos outputs para uso no Script 5 (cenários LLM).

Produz:
- Intervalo preditivo [P5, P95] por empresa × target
- Coeficiente de Variação (CV) — sensibilidade do modelo
- P(predição < 0) para targets que podem ser negativos (Lucro, FCO)
- `stress_empresas_destaque` — mapa das empresas destacadas por setor

In [ ]:
EMPRESAS_DESTAQUE_OVERRIDE = {}

rng = np.random.default_rng(SEED_MC)
res_stress  = []
TARGETS_STRESS = [t for t in TARGETS if t.endswith('_DFP')]

if 'CNPJ_CIA' in dataset.columns and 'NOME_CIA' in dataset.columns:
    todas_empresas = (dataset[['CNPJ_CIA','NOME_CIA','SETOR']]
                      .drop_duplicates('CNPJ_CIA')
                      .dropna(subset=['NOME_CIA'])
                      .to_dict('records'))
else:
    todas_empresas = []
    print('AVISO: CNPJ_CIA ou NOME_CIA ausente — Monte Carlo ignorado.')

def _ultima_linha(cnpj):
    df_e = dataset[dataset['CNPJ_CIA'] == cnpj]
    if df_e.empty: return None, None
    if 'ANO' in df_e.columns: df_e = df_e.sort_values('ANO')
    ul  = df_e.iloc[-1]
    ano = int(ul['ANO']) if 'ANO' in ul.index else None
    return ul, ano

print(f'Monte Carlo: {len(todas_empresas)} empresas x '
      f'{len(TARGETS_STRESS)} targets x {N_SIM_MC} simulacoes')

for emp_info in todas_empresas:
    cnpj     = emp_info['CNPJ_CIA']
    nome_emp = emp_info['NOME_CIA']
    setor    = emp_info.get('SETOR', 'N/D')

    row_base, ano_base = _ultima_linha(cnpj)
    if row_base is None: continue

    for target in TARGETS_STRESS:
        if target not in melhores: continue
        alg_nome = melhores[target]
        feats    = selected_features_por_target.get(target, [])
        feats    = [f for f in feats if f in dataset.columns]
        if not feats: continue

        cam = PASTA_SAIDA / 'modelos' / f'modelo_{target}_{alg_nome}.pkl'
        if not cam.exists(): continue
        try:
            obj    = joblib.load(cam)
            modelo = obj['modelo'] if isinstance(obj, dict) else obj
        except Exception: continue

        transf = get_transform(target)

        # Correcao BUG 1: `float(v or 0)` nao funciona com np.nan.
        # np.nan e truthy em Python: `np.nan or 0` retorna np.nan, nao 0.
        # Usa pd.notna() para verificacao explicita.
        x_base_vals = []
        for f in feats:
            v = row_base.get(f)
            x_base_vals.append(float(v) if pd.notna(v) else 0.0)
        x_base = np.array(x_base_vals, dtype=float)

        # Correcao BUG 2: `array != None` conta NaN como valido
        # porque np.nan != None e True em Python.
        # Usa np.isfinite() que retorna False para NaN e Inf.
        n_feats_validos = int(np.sum(np.isfinite(x_base)))

        ruido  = rng.normal(0, SIGMA_MC * (np.abs(x_base) + 1e-9),
                            size=(N_SIM_MC, len(feats)))
        X_sim  = x_base[None, :] + ruido

        try:
            y_sim  = inv_transform(modelo.predict(X_sim), transf)
            y_base = float(inv_transform(modelo.predict(x_base[None, :]), transf)[0])
        except Exception: continue

        p5, p50, p95 = np.percentile(y_sim, [5, 50, 95])
        media_mc = float(np.mean(y_sim))
        std_mc   = float(np.std(y_sim))
        cv       = float(std_mc / abs(media_mc)) if abs(media_mc) > 1e-9 else np.nan
        p_neg    = (float(np.mean(y_sim < 0))
                    if any(b in target for b in ['DRE_3.11','DFC_MI']) else np.nan)

        base_str = target.replace('TARGET_', '').replace('_DFP', '')
        res_stress.append({
            'cnpj': cnpj, 'empresa': nome_emp, 'setor': setor,
            'ano_base': ano_base, 'target': target,
            'variavel': NOME_VARIAVEL.get(base_str, base_str),
            'algoritmo': alg_nome, 'n_feats_validos': n_feats_validos,
            'y_base': y_base, 'media_mc': media_mc, 'std_mc': std_mc,
            'cv': cv, 'p5': p5, 'p50': p50, 'p95': p95,
            'p_negativo': p_neg, 'n_sim': N_SIM_MC, 'sigma': SIGMA_MC,
        })

if res_stress:
    df_stress = pd.DataFrame(res_stress)
    df_stress.to_csv(PASTA_SAIDA / 'analise_estresse_mc.csv', index=False)

    print(f'\nOK Monte Carlo: {len(df_stress)} combinacoes empresa x target')
    print(f'   Empresas: {df_stress["empresa"].nunique()} | '
          f'Setores: {sorted(df_stress["setor"].unique())}')

    EMPRESAS_DESTAQUE = {}
    if EMPRESAS_DESTAQUE_OVERRIDE:
        EMPRESAS_DESTAQUE = EMPRESAS_DESTAQUE_OVERRIDE
        print('\nEmpresas destaque (manual):')
    else:
        df_crit = df_stress[df_stress['target'].str.contains('DRE_3.01_DFP', na=False)]
        for setor, grp in df_crit.groupby('setor'):
            melhor = grp.sort_values('n_feats_validos', ascending=False).iloc[0]
            EMPRESAS_DESTAQUE[setor] = {'nome': melhor['empresa'], 'cnpj': melhor['cnpj']}
        print('\nEmpresas destaque por setor (selecao automatica):')

    for setor, info in EMPRESAS_DESTAQUE.items():
        nome = info['nome'] if isinstance(info, dict) else info
        print(f'  {setor:<20} -> {nome}')

    import pickle
    with open(PASTA_SAIDA / 'empresas_destaque.pkl', 'wb') as fh:
        pickle.dump(EMPRESAS_DESTAQUE, fh)
    print('\nOK empresas_destaque.pkl salvo')
else:
    df_stress = pd.DataFrame()
    EMPRESAS_DESTAQUE = {}
    print('AVISO: Estresse MC sem resultados.')

print('\nOK Etapa 8 concluida')


## Etapa 8b. Intervalos de Predição — Conformal Prediction vs. Monte Carlo

**Por que aqui?** O Monte Carlo (Etapa 8) mede sensibilidade das *entradas*.
O Conformal mede incerteza dos *erros reais* do modelo. São complementares.

**Protocolo: Split Conformal (Papadopoulos et al., 2002; Angelopoulos & Bates, 2021)**

1. Hold-out 2024–2025 dividido: 2024 → calibração | 2025 → teste final
2. Score de não-conformidade: `s_i = |y_true_i − y_pred_i|` (escala original)
3. Quantil calibrado: `q̂ = Quantile(s_calib, ⌈(n+1)(1−α)/n⌉)` → garante cobertura finita
4. Intervalo: `[ŷ − q̂, ŷ + q̂]` com `P(y_true ∈ IC) ≥ 1−α` **sem suposição distribucional**

**Diferença fundamental do Monte Carlo:**
- Monte Carlo σ=15%: "como a predição varia se os KPIs oscilarem 15%?" → análise de estresse
- Conformal: "onde o valor real estará com 90% de garantia?" → intervalo de predição calibrado

**Saídas:** `conformal_intervals.parquet` | `conformal_summary.csv` | `conformal_vs_mc.png`

In [ ]:
# =============================================================================
# Etapa 8b — Split Conformal Prediction
#
# Implementação nativa — sem dependência extra (não requer mapie).
# Funciona sobre os modelos já treinados pelo Script 3 sem nenhum retreino.
#
# Garantia matemática (Vovk et al., 2005):
#   Para qualquer distribuição (x, y) e qualquer modelo f:
#   P(y_test ∈ [f(x_test) - q̂, f(x_test) + q̂]) >= 1 - alpha
#   contanto que (x_calib, y_calib) e (x_test, y_test) sejam trocáveis.
# =============================================================================

ANO_CALIB = 2024
ANO_TEST2 = 2025
ALPHA_CP  = 0.10   # IC 90%

def conformal_interval(yc_true, yc_pred, yt_pred, alpha=0.10):
    """
    Split Conformal com correção de cobertura finita.
    Retorna (lb, ub, q_hat) na mesma escala dos inputs.
    """
    scores = np.abs(np.asarray(yc_true, float) - np.asarray(yc_pred, float))
    n      = len(scores)
    level  = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    q_hat  = float(np.quantile(scores, level))
    yt     = np.asarray(yt_pred, float)
    return yt - q_hat, yt + q_hat, q_hat

# ── Split do hold-out ─────────────────────────────────────────────────────────
if 'ANO' in teste.columns:
    calib = teste[teste['ANO'] == ANO_CALIB].copy()
    test2 = teste[teste['ANO'] == ANO_TEST2].copy()
    print(f'Calibração (2024): {len(calib):,} obs | Teste final (2025): {len(test2):,} obs')
else:
    np.random.seed(42)
    idx_c = np.random.choice(len(teste), len(teste)//2, replace=False)
    mask  = np.zeros(len(teste), dtype=bool); mask[idx_c] = True
    calib = teste[mask].copy()
    test2 = teste[~mask].copy()
    print(f'⚠️  ANO ausente — split 50/50 aleatório')

rows_cp = []

for target in TARGETS:
    if target not in melhores:
        continue
    alg_nome = melhores[target]

    # Carrega modelo do disco
    cam = PASTA_SAIDA / 'modelos' / f'modelo_{target}_{alg_nome}.pkl'
    if not cam.exists():
        continue
    try:
        obj    = joblib.load(cam)
        modelo = obj['modelo'] if isinstance(obj, dict) else obj
    except Exception:
        continue

    feats = selected_features_por_target.get(target, [])
    feats = [f for f in feats if f in calib.columns and f in test2.columns]
    if not feats:
        continue

    transf = get_transform(target)

    # ── Predições na calibração (escala original) ─────────────────────────
    df_c = calib[[f for f in feats] + [target]].dropna(subset=[target]).copy()
    if df_c.empty:
        continue
    yc_raw  = modelo.predict(df_c[feats].values)
    yc_pred = inv_transform(yc_raw, transf)
    yc_true = inv_transform(df_c[target].values, transf)

    # ── Predições no teste final (escala original) ────────────────────────
    meta_cols = [c for c in ['CNPJ_CIA', 'DT_REFER', 'ANO'] if c in test2.columns]
    df_t2 = test2[[f for f in feats] + [target] + meta_cols].dropna(subset=[target]).copy()
    if df_t2.empty:
        continue
    yt2_raw  = modelo.predict(df_t2[feats].values)
    yt2_pred = inv_transform(yt2_raw, transf)
    yt2_true = inv_transform(df_t2[target].values, transf)

    # ── Intervalo conformal ───────────────────────────────────────────────
    lb, ub, q_hat = conformal_interval(yc_true, yc_pred, yt2_pred, alpha=ALPHA_CP)
    coberto       = (yt2_true >= lb) & (yt2_true <= ub)
    cob_real      = float(coberto.mean())
    largura_media = float((ub - lb).mean())

    base = target.replace('TARGET_','').rsplit('_ITR',1)[0].rsplit('_DFP',1)[0]
    hor  = next((h for h in _HORIZONTES if target.endswith(h)), 'N/A')

    for i in range(len(df_t2)):
        row = df_t2.iloc[i]
        rows_cp.append({
            'CNPJ_CIA':  row.get('CNPJ_CIA'),
            'DT_REFER':  row.get('DT_REFER'),
            'ANO':       row.get('ANO'),
            'Target':    target,
            'Variavel':  NOME_VARIAVEL.get(base, base),
            'Horizonte': hor.replace('_',''),
            'Algoritmo': alg_nome,
            'y_true':    yt2_true[i],
            'y_pred':    yt2_pred[i],
            'lb_90':     lb[i],
            'ub_90':     ub[i],
            'width_90':  float(ub[i] - lb[i]),
            'coberto':   bool(coberto[i]),
            'q_hat':     q_hat,
            'alpha':     ALPHA_CP,
        })

    print(f'{NOME_VARIAVEL.get(base,base):<22} [{hor.replace("_","")}] {alg_nome:<18} '
          f'q̂={q_hat/1e6:>7.1f}Bi  Cob={cob_real:.1%}  Larg={largura_media/1e6:>7.1f}Bi')

if rows_cp:
    df_cp = pd.DataFrame(rows_cp)
    df_cp.to_parquet(PASTA_SAIDA / 'conformal_intervals.parquet', index=False)
    df_cp.to_csv(PASTA_SAIDA    / 'conformal_intervals.csv',      index=False)

    # Resumo por variável e horizonte
    resumo_cp = df_cp.groupby(['Variavel','Horizonte']).agg(
        cobertura_real =('coberto',   'mean'),
        largura_media  =('width_90',  'mean'),
        q_hat          =('q_hat',     'first'),
        n              =('coberto',   'count'),
    ).reset_index()
    resumo_cp.to_csv(PASTA_SAIDA / 'conformal_summary.csv', index=False)

    print(f'\n=== Resumo IC {int((1-ALPHA_CP)*100)}% — Conformal Prediction ===')
    print(resumo_cp.round(4).to_string(index=False))

    # ── Figura: Conformal vs. Monte Carlo — Receita Líquida DFP ──────────
    _t_ex = next((t for t in TARGETS if t.endswith('_DFP') and 'DRE_3.01' in t), None)
    if _t_ex and not df_cp.empty:
        df_ex = df_cp[df_cp['Target'] == _t_ex].sort_values('y_true').reset_index(drop=True)

        # Dados do Monte Carlo para comparação
        mc_row = df_stress[df_stress['target'] == _t_ex] if not df_stress.empty else pd.DataFrame()

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Painel esquerdo — cobertura conformal
        ax = axes[0]
        x  = np.arange(len(df_ex))
        ax.fill_between(x, df_ex['lb_90']/1e6, df_ex['ub_90']/1e6,
                        alpha=0.30, color='#3498db', label='IC Conformal 90%')
        ax.plot(x, df_ex['y_pred']/1e6, color='#3498db', lw=1.5, label='Predição')
        fora = ~df_ex['coberto']
        dentro = df_ex['coberto']
        ax.scatter(x[fora],   df_ex.loc[fora,   'y_true']/1e6,
                   color='#e74c3c', s=20, zorder=5, label='Fora do IC')
        ax.scatter(x[dentro], df_ex.loc[dentro, 'y_true']/1e6,
                   color='#2ecc71', s=10, alpha=0.5, label='Dentro do IC')
        cob_real = df_ex['coberto'].mean()
        ax.set_title(f'Conformal Prediction — Receita Líquida (DFP)\n'
                     f'Cobertura real: {cob_real:.1%} (alvo: {int((1-ALPHA_CP)*100)}%)',
                     fontsize=10, fontweight='bold')
        ax.set_xlabel('Observação (ordenada por y_true)')
        ax.set_ylabel('R$ bilhões')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

        # Painel direito — largura: Conformal vs. IC Monte Carlo (P5–P95)
        ax2 = axes[1]
        larg_cp  = df_ex['width_90'].mean() / 1e6
        # MC: média de (p95 - p5) para Receita Líquida DFP
        if not mc_row.empty:
            larg_mc = float((mc_row['p95'] - mc_row['p5']).mean()) / 1e6
            labels  = ['Conformal\n(garantia 90%)', 'Monte Carlo\n(P5–P95, σ=15%)']
            valores = [larg_cp, larg_mc]
            cores_b = ['#3498db', '#e74c3c']
        else:
            resid_std = (df_ex['y_true'] - df_ex['y_pred']).std() / 1e6
            larg_naive = 4 * resid_std
            labels  = ['Conformal\n(garantia 90%)', 'Naive\n(±2σ)']
            valores = [larg_cp, larg_naive]
            cores_b = ['#3498db', '#95a5a6']

        bars2 = ax2.bar(labels, valores, color=cores_b, edgecolor='white')
        for bar, v in zip(bars2, valores):
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f'{v:.2f} Bi', ha='center', fontsize=10)
        ax2.set_ylabel('Largura média do IC (R$ bilhões)')
        ax2.set_title('Largura do Intervalo:\nConformal vs. Monte Carlo',
                      fontsize=10, fontweight='bold')
        ax2.grid(axis='y', alpha=0.3)

        plt.suptitle('Análise de Incerteza — Conformal Prediction vs. Monte Carlo\n'
                     '(Receita Líquida, Horizonte DFP, Hold-out 2025)',
                     fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(PASTA_SAIDA / 'figuras' / 'conformal_vs_mc.png',
                    dpi=150, bbox_inches='tight')
        plt.close()
        print('✅ conformal_vs_mc.png salvo')

    print(f'\n✅ Etapa 8b concluída — {len(rows_cp)} intervalos conformais | '
          f'IC {int((1-ALPHA_CP)*100)}%')
    print('   Arquivos: conformal_intervals.parquet | conformal_summary.csv | '
          'figuras/conformal_vs_mc.png')
else:
    df_cp = pd.DataFrame()
    print('⚠️  Nenhum intervalo conformal calculado (modelos ou split ausentes).')

## Etapa 9. Feature Importance Agregada com Ranking por Família

In [ ]:
df_feat_rank = pd.DataFrame()

if feature_importances:
    acum={}
    for target, info in feature_importances.items():
        if not info or 'importancias' not in info: continue
        for feat, val in info['importancias'].items():
            if pd.notna(val): acum[feat]=acum.get(feat,0.0)+float(val)

    df_feat_rank = (pd.Series(acum).reset_index()
                    .rename(columns={'index':'feature',0:'importancia_total'})
                    .sort_values('importancia_total',ascending=False)
                    .reset_index(drop=True))
    df_feat_rank['rank'] = df_feat_rank.index+1

    def familia(f):
        if f.startswith('macro_'):                   return 'Macro'
        if '_yoy' in f:                              return 'YoY'
        if '_lag' in f or '_roll' in f:              return 'Lag/Roll'
        if f.startswith('ratio_') or '_over_' in f: return 'Razão/Cruzada'
        if f.startswith('pos_setor_'):               return 'Posição Setor'
        if f.startswith('setor_'):                   return 'Setor dummy'
        if f.startswith('tri_'):                     return 'Sazonalidade'
        if f in ['flag_dfp','flag_covid','ano_norm']:return 'Temporal'
        if f in (KPIS or []):                        return 'KPI base'
        return 'Outro'

    df_feat_rank['familia'] = df_feat_rank['feature'].apply(familia)
    df_feat_rank.to_csv(PASTA_SAIDA/'feature_importance_ranking.csv', index=False)

    print('=== Top-25 Features por Importância Agregada (todos os 36 targets) ===')
    print(df_feat_rank.head(25)[['rank','feature','familia','importancia_total']].to_string(index=False))
    print('\nImportância total por família:')
    print(df_feat_rank.groupby('familia')['importancia_total']
          .agg(['sum','count','mean']).sort_values('sum',ascending=False).round(4).to_string())
else:
    print('⚠️  feature_importances.pkl não disponível.')

print('\n✅ Etapa 9 concluída')

## Etapa 10. Análise de Resíduos e Teste de Viés Sistemático

In [ ]:
if not df_pred.empty:
    df_res_best = df_pred[
        df_pred.apply(lambda r: melhores.get(r['Target']) == r['Algoritmo'], axis=1)
    ].copy()

    if not df_res_best.empty:
        # Script 3 salva y_true em escala TRANSFORMADA e y_pred em escala ORIGINAL.
        # Sem inversao: y_true~20 (log), y_pred~1e9 (R$) -> denom~5e8 -> erro_sim~0.
        # Correcao: inverte y_true para R$ antes de calcular residuos.
        def _inv_ytrue(row):
            v = row['y_true']
            if not np.isfinite(v): return np.nan
            return float(inv_transform([v], get_transform(row['Target']))[0])

        df_res_best['y_true_orig'] = df_res_best.apply(_inv_ytrue, axis=1)
        df_res_best['y_pred_orig'] = df_res_best['y_pred'].astype(float)

        denom = (df_res_best['y_true_orig'].abs() + df_res_best['y_pred_orig'].abs()) / 2.0
        df_res_best['erro_sim'] = np.where(
            denom > 1e-9,
            (df_res_best['y_true_orig'] - df_res_best['y_pred_orig']).abs() / denom,
            np.nan
        )

        df_res_best['horizonte'] = df_res_best['Target'].apply(
            lambda t: next((h for h in _HORIZONTES if t.endswith(h)), 'N/A'))
        df_res_best['base_variavel'] = df_res_best['Target'].apply(
            lambda t: t.replace('TARGET_','').rsplit('_ITR',1)[0].rsplit('_DFP',1)[0])
        df_res_best['variavel']  = df_res_best['base_variavel'].map(NOME_VARIAVEL).fillna(df_res_best['base_variavel'])
        df_res_best['NOME_CIA']  = df_res_best['CNPJ_CIA'].map(mapa_nome)

        print('=== Estatisticas de Residuos (todos os targets, melhor modelo) ===')
        stats = (df_res_best.groupby(['variavel','horizonte'])['erro_sim']
                 .agg(media='mean', mediana='median', std='std',
                      p5=lambda x: np.percentile(x.dropna(), 5)  if x.dropna().size > 0 else np.nan,
                      p95=lambda x: np.percentile(x.dropna(), 95) if x.dropna().size > 0 else np.nan)
                 .round(4))
        print(stats.to_string())

        print('\n=== Teste de vies sistematico (H0: SMAPE medio = 0) ===')
        for target in TARGETS:
            sub = df_res_best[df_res_best['Target'] == target]['erro_sim'].dropna()
            if len(sub) < 5: continue
            t_stat, p_val = sp_stats.ttest_1samp(sub, 0)
            flag = ' VIES' if p_val < 0.05 else ''
            base = target.replace('TARGET_','').rsplit('_ITR',1)[0].rsplit('_DFP',1)[0]
            hor  = next((h for h in _HORIZONTES if target.endswith(h)), '')
            print(f'  {NOME_VARIAVEL.get(base,base):<22} [{hor.replace("_","")}]  '
                  f't={t_stat:+.3f}  p={p_val:.4f}{flag}')

        piores = (df_res_best.groupby(['CNPJ_CIA','NOME_CIA'])['erro_sim']
                  .mean().nlargest(5))
        print('\nTop-5 empresas com maior SMAPE medio:')
        print(piores.round(4).to_string())

        df_res_best.to_parquet(PASTA_SAIDA / 'residuos_detalhados.parquet', index=False)
        print(f'\nOK residuos_detalhados.parquet salvo ({len(df_res_best):,} linhas)')
else:
    print('AVISO: predicoes_teste_detalhadas.parquet nao disponivel.')

print('\nOK Etapa 10 concluida')


## Etapa 11. Visualizações (8 figuras)

In [ ]:
PALETA_ALG = {'Ridge':'#3498db','SVR':'#9b59b6',
               'RandomForest':'#2ecc71','GradientBoosting':'#e74c3c'}
CORES_FAM  = {'Macro':'#e74c3c','YoY':'#2ecc71','Lag/Roll':'#3498db',
              'KPI base':'#f39c12','Razão/Cruzada':'#1abc9c',
              'Setor dummy':'#9b59b6','Posição Setor':'#e67e22',
              'Sazonalidade':'#95a5a6','Temporal':'#bdc3c7','Outro':'#ecf0f1'}

# Fig 1 — Heatmap SMAPE por Algoritmo × Horizonte
if not df_te.empty and 'Horizonte' in df_te.columns:
    try:
        _c = next((c for c in df_te.columns if 'SMAPE_teste' in c), None)
        if _c:
            pv = df_te.groupby(['Algoritmo','Horizonte'])[_c].mean().unstack()
            fig, ax = plt.subplots(figsize=(9,4))
            sns.heatmap(pv, annot=True, fmt='.3f', cmap='RdYlGn_r',
                        linewidths=0.5, linecolor='white', ax=ax,
                        cbar_kws={'label':'SMAPE macro'})
            ax.set_title('SMAPE macro por Algoritmo × Horizonte (36 targets)',
                         fontsize=11, fontweight='bold')
            plt.tight_layout()
            plt.savefig(PASTA_SAIDA/'figuras'/'heatmap_smape_36targets.png',
                        dpi=150, bbox_inches='tight')
            plt.close(); print('✅ Fig 1')
    except Exception as e: print(f'⚠️ Fig 1: {e}')

# Fig 2 — Heatmap SMAPE por Variável × Algoritmo (DFP)
if not df_te.empty:
    try:
        _c = next((c for c in df_te.columns if 'SMAPE_teste' in c), None)
        if _c and 'base_variavel' in df_te.columns:
            df_dfp = df_te[df_te['Horizonte']=='_DFP'] if 'Horizonte' in df_te.columns else df_te
            pv = df_dfp.groupby(['base_variavel','Algoritmo'])[_c].mean().unstack()
            pv.index = [NOME_VARIAVEL.get(i,i) for i in pv.index]
            fig, ax = plt.subplots(figsize=(11,5))
            sns.heatmap(pv, annot=True, fmt='.3f', cmap='RdYlGn_r',
                        linewidths=0.5, linecolor='white', ax=ax,
                        cbar_kws={'label':'SMAPE macro (DFP)'})
            ax.set_title('SMAPE por Variável × Algoritmo — Horizonte DFP',
                         fontsize=11, fontweight='bold')
            plt.tight_layout()
            plt.savefig(PASTA_SAIDA/'figuras'/'heatmap_smape_variaveis_dfp.png',
                        dpi=150, bbox_inches='tight')
            plt.close(); print('✅ Fig 2')
    except Exception as e: print(f'⚠️ Fig 2: {e}')

# Fig 3 — Distribuição Z\'\' por setor
if not df_zscore.empty and 'SETOR' in df_zscore.columns:
    try:
        df_zp    = df_zscore[df_zscore['altman_z_pp'].notna()]
        setores  = sorted(df_zp['SETOR'].dropna().unique())
        fig, axes = plt.subplots(1, len(setores),
                                  figsize=(4*len(setores), 5), sharey=False)
        if len(setores) == 1: axes = [axes]
        for ax, s in zip(axes, setores):
            d = df_zp[df_zp['SETOR']==s]['altman_z_pp'].dropna()
            ax.hist(d, bins=15, color='#3498db', edgecolor='white', alpha=0.85)
            ax.axvline(ZONA_CINZA_INF, color='#e74c3c', ls='--', lw=1.5)
            ax.axvline(ZONA_SEGURA,   color='#2ecc71', ls='--', lw=1.5)
            ax.set_title(s, fontsize=10, fontweight='bold')
            ax.set_xlabel("Z\'\' ")
        plt.suptitle("Distribuição Z\'\' por Setor", fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(PASTA_SAIDA/'figuras'/'dist_zscore_setor.png',
                    dpi=150, bbox_inches='tight')
        plt.close(); print('✅ Fig 3')
    except Exception as e: print(f'⚠️ Fig 3: {e}')

# Fig 4 — Score de Risco por Setor
if 'score_risco' in dataset.columns and 'SETOR' in dataset.columns:
    try:
        ord_s = dataset.groupby('SETOR')['score_risco'].median().sort_values(ascending=False).index
        fig, ax = plt.subplots(figsize=(9,5))
        sns.boxplot(data=dataset[dataset['score_risco'].notna()],
                    x='SETOR', y='score_risco', order=ord_s,
                    palette='RdYlGn_r', ax=ax)
        for lim, cor, lbl in [(20,'#2ecc71','Baixo'),(40,'#f39c12','Moderado'),(60,'#e74c3c','Elevado')]:
            ax.axhline(lim, ls=':', color=cor, lw=1.5, alpha=0.7, label=f'{lbl} ({lim})')
        ax.set_title('Score de Risco Composto por Setor',
                     fontsize=12, fontweight='bold')
        ax.legend(fontsize=8); plt.tight_layout()
        plt.savefig(PASTA_SAIDA/'figuras'/'score_risco_setor.png',
                    dpi=150, bbox_inches='tight')
        plt.close(); print('✅ Fig 4')
    except Exception as e: print(f'⚠️ Fig 4: {e}')

# Fig 5 — Feature Importance Top-25
if not df_feat_rank.empty:
    try:
        top  = df_feat_rank.head(25)
        cores = top['familia'].map(CORES_FAM).fillna('#bdc3c7')
        fig, ax = plt.subplots(figsize=(12,8))
        ax.barh(range(len(top)), top['importancia_total'].values[::-1],
                color=cores.values[::-1])
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'].values[::-1], fontsize=8)
        ax.set_xlabel('Importância Agregada (36 targets)', fontsize=10)
        ax.set_title('Top-25 Features por Importância Agregada',
                     fontsize=12, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)
        patches = [mpatches.Patch(color=v, label=k)
                   for k,v in CORES_FAM.items() if k in top['familia'].values]
        ax.legend(handles=patches, fontsize=8, loc='lower right')
        plt.tight_layout()
        plt.savefig(PASTA_SAIDA/'figuras'/'feature_importance_top25.png',
                    dpi=150, bbox_inches='tight')
        plt.close(); print('✅ Fig 5')
    except Exception as e: print(f'⚠️ Fig 5: {e}')

# Fig 6 — Evolução Z\'\' — TODAS as empresas; destaque nas representativas
if not df_zscore.empty and 'ANO' in df_zscore.columns and 'NOME_CIA' in df_zscore.columns:
    try:
        # Lista de todas as empresas com Z'' válido
        emps_all  = (df_zscore[df_zscore['altman_z_pp'].notna()]
                     ['NOME_CIA'].dropna().unique().tolist())
        # Empresas destaque por setor (do Script 4 MC ou fallback)
        emps_dest = ([v['nome'] if isinstance(v, dict) else v
                      for v in EMPRESAS_DESTAQUE.values()]
                     if 'EMPRESAS_DESTAQUE' in dir() and EMPRESAS_DESTAQUE
                     else emps_all[:5])

        df_ze = df_zscore[
            df_zscore['NOME_CIA'].isin(emps_all) &
            df_zscore['altman_z_pp'].notna()
        ]
        if not df_ze.empty:
            fig, ax = plt.subplots(figsize=(13, 6))
            cs = plt.cm.tab20(np.linspace(0, 1, len(emps_all)))
            for i, emp in enumerate(emps_all):
                s = df_ze[df_ze['NOME_CIA'] == emp].sort_values('ANO')
                if s.empty: continue
                destaque = emp in emps_dest
                ax.plot(s['ANO'], s['altman_z_pp'],
                        marker='o' if destaque else None,
                        label=emp if destaque else '_nolegend_',
                        color=cs[i % len(cs)],
                        linewidth=2.5 if destaque else 1.0,
                        markersize=5 if destaque else 0,
                        alpha=1.0 if destaque else 0.35,
                        zorder=5 if destaque else 2)
                if destaque:
                    ax.annotate(emp[:14],
                                xy=(s['ANO'].iloc[-1], s['altman_z_pp'].iloc[-1]),
                                xytext=(4, 2), textcoords='offset points',
                                fontsize=6, color=cs[i % len(cs)],
                                fontweight='bold')
            ax.axhspan(-10, ZONA_CINZA_INF, alpha=0.06, color='#e74c3c')
            ax.axhspan(ZONA_CINZA_INF, ZONA_SEGURA, alpha=0.06, color='#f39c12')
            ax.axhspan(ZONA_SEGURA, 30, alpha=0.04, color='#2ecc71')
            ax.axhline(ZONA_CINZA_INF, color='#e74c3c', ls='--', lw=1.2)
            ax.axhline(ZONA_SEGURA,   color='#2ecc71', ls='--', lw=1.2)
            ax.set_xlabel('Ano'); ax.set_ylabel("Z\'\' ")
            ax.set_title(f"Evolução Z\'\' — Todas as Empresas (2015–2025)\n"
                         f"Linhas destacadas = empresas representativas por setor",
                         fontsize=11, fontweight='bold')
            ax.legend(fontsize=7, ncol=3, loc='upper left')
            ax.grid(alpha=0.3)
            plt.tight_layout()
            plt.savefig(PASTA_SAIDA/'figuras'/'evolucao_zscore_todas.png',
                        dpi=150, bbox_inches='tight')
            plt.close(); print(f'✅ Fig 6 ({len(emps_all)} empresas, {len(emps_dest)} destacadas)')
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f'⚠️ Fig 6: {e}')

# Fig 7 — Predito × Observado (escala original — inversão de transformação)
if not df_pred.empty:
    try:
        df_fp = df_pred[
            df_pred['Target'].isin(TARGETS_FOCO) &
            df_pred.apply(lambda r: melhores.get(r['Target'])==r['Algoritmo'], axis=1)
        ].copy()
        tgts_plot = [t for t in TARGETS_FOCO if t in df_fp['Target'].values][:12]
        if tgts_plot:
            ncols = 4; nrows = int(np.ceil(len(tgts_plot)/ncols))
            fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows))
            axes = axes.flatten()
            for idx, target in enumerate(tgts_plot):
                ax = axes[idx]
                sub = df_fp[df_fp['Target']==target].dropna(subset=['y_true','y_pred'])
                if sub.empty: ax.set_visible(False); continue
                # Inversão ANTES de plotar
                transf = get_transform(target)
                # Script 3: y_true em escala transformada, y_pred ja em R$.
                # inv_transform apenas em y_true; y_pred usado diretamente.
                yt = inv_transform(sub['y_true'].values, transf)
                yp = sub['y_pred'].values.astype(float)
                alg = melhores.get(target,'?')
                r2v = _r2(yt,yp); spv = _smape(yt,yp)
                ax.scatter(yp, yt, alpha=0.5, s=18, edgecolors='none',
                           color=PALETA_ALG.get(alg,'#3498db'))
                lm = min(yt.min(),yp.min()); lM = max(yt.max(),yp.max())
                ax.plot([lm,lM],[lm,lM],'k--',lw=1)
                base = target.replace('TARGET_','').rsplit('_ITR',1)[0].rsplit('_DFP',1)[0]
                hor  = next((h.replace('_','') for h in _HORIZONTES if target.endswith(h)),'')
                ax.set_title(f'{NOME_VARIAVEL.get(base,base)} [{hor}]\n'
                             f'{alg}  R²={r2v:.3f}  SMAPE={spv:.1%}', fontsize=8)
                ax.set_xlabel('Predito (R$ mil)', fontsize=7)
                ax.set_ylabel('Observado (R$ mil)', fontsize=7)
                ax.tick_params(labelsize=7)
                if np.nanmax(np.abs(yt)) > 1e6:
                    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x/1e6:.1f}Bi'))
                    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x/1e6:.1f}Bi'))
            for j in range(idx+1, len(axes)): axes[j].set_visible(False)
            plt.suptitle('Predito × Observado — Targets Foco TCC\n(escala original R$ mil)',
                         fontsize=12, fontweight='bold')
            plt.tight_layout()
            plt.savefig(PASTA_SAIDA/'figuras'/'predito_vs_observado_foco.png',
                        dpi=150, bbox_inches='tight')
            plt.close(); print('✅ Fig 7')
    except Exception as e: print(f'⚠️ Fig 7: {e}')

# Fig 8 — Intervalos Monte Carlo P5–P95 — todas as empresas por variável
if not df_stress.empty:
    try:
        for base_str in ['DRE_3.01','DRE_3.11','EBITDA']:
            t_alvo = f'TARGET_{base_str}_DFP'
            df_sf  = df_stress[df_stress['target'] == t_alvo].copy()
            if df_sf.empty: continue
            df_sf = df_sf.sort_values(['setor','empresa']).reset_index(drop=True)
            n     = len(df_sf)
            emps_dest_nomes = ([v['nome'] if isinstance(v,dict) else v
                                for v in EMPRESAS_DESTAQUE.values()]
                               if 'EMPRESAS_DESTAQUE' in dir() and EMPRESAS_DESTAQUE else [])

            fig, ax = plt.subplots(figsize=(max(12, 0.6*n), 5))
            x = np.arange(n)
            cores_bar = ['#e74c3c' if row['empresa'] in emps_dest_nomes
                         else '#3498db'
                         for _, row in df_sf.iterrows()]
            ax.bar(x, df_sf['p50']/1e6, color=cores_bar, alpha=0.7, label='Mediana MC')
            ax.errorbar(x, df_sf['p50']/1e6,
                        yerr=[(df_sf['p50']-df_sf['p5'])/1e6,
                              (df_sf['p95']-df_sf['p50'])/1e6],
                        fmt='none', color='#2c3e50', capsize=3, linewidth=1.2,
                        label='IC [P5, P95]')
            ax.scatter(x, df_sf['y_base']/1e6, color='#f39c12',
                       zorder=5, s=40, label='Base (sem perturbação)')
            ax.set_xticks(x)
            ax.set_xticklabels(
                [e[:12]+'…' if len(e)>12 else e for e in df_sf['empresa']],
                rotation=45, ha='right', fontsize=7)
            ax.set_ylabel('R$ bilhões', fontsize=9)
            ax.set_title(f'Intervalos MC P5–P95 — {NOME_VARIAVEL.get(base_str,base_str)} DFP 2026\n'
                         f'Barras vermelhas = empresas representativas por setor',
                         fontsize=10, fontweight='bold')
            ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
            # Adiciona label de setor no eixo x por grupo
            setores_unicos = df_sf['setor'].unique()
            for s in setores_unicos:
                idx_s = df_sf[df_sf['setor']==s].index
                mid   = (idx_s[0] + idx_s[-1]) / 2
                ax.text(mid, ax.get_ylim()[0]*0.95, s[:10],
                        ha='center', fontsize=7, color='#555',
                        fontstyle='italic', transform=ax.transData)
            plt.tight_layout()
            fname = f'intervalos_mc_{base_str.replace(".","_")}.png'
            plt.savefig(PASTA_SAIDA/'figuras'/fname, dpi=150, bbox_inches='tight')
            plt.close()
            print(f'✅ Fig 8 ({NOME_VARIAVEL.get(base_str,base_str)}): {fname}')
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f'⚠️ Fig 8: {e}')

print('\n✅ Etapa 11 concluída — 8 grupos de figuras gerados')

## Etapa 12. Persistência Completa de Artefatos para o Script 5

In [ ]:
# 12.1 Melhores modelos confirmados
with open(PASTA_SAIDA/'melhores_modelos_v4.pkl','wb') as f: pickle.dump(melhores,f)

# 12.2 Z-Score por empresa (resumo para Script 5)
if not df_zscore.empty:
    zpe={}
    _ca='ANO' if 'ANO' in df_zscore.columns else None
    for cnpj, grp in df_zscore[df_zscore['altman_z_pp'].notna()].groupby('CNPJ_CIA'):
        grp_s=grp.sort_values(_ca) if _ca else grp
        ul=grp_s.iloc[-1]
        zpe[cnpj]={'z_medio':float(grp['altman_z_pp'].mean()),
                   'z_ultimo':float(ul['altman_z_pp']),
                   'zona_ultimo':str(ul['zona_altman']),
                   'nome':str(ul.get('NOME_CIA',cnpj)),
                   'setor':str(ul.get('SETOR',''))}
    with open(PASTA_SAIDA/'zscore_por_empresa.pkl','wb') as f: pickle.dump(zpe,f)
    print(f'✅ zscore_por_empresa.pkl: {len(zpe)} empresas')

# 12.3 Score de risco do dataset
if 'score_risco' in dataset.columns:
    _cr=[c for c in ['CNPJ_CIA','NOME_CIA','ANO','SETOR',
                     'score_risco','classe_risco','altman_z_pp','zona_altman']
         if c in dataset.columns]
    dataset[_cr].to_parquet(PASTA_SAIDA/'score_risco_dataset.parquet',index=False)
    print('✅ score_risco_dataset.parquet salvo')

# 12.4 Feature importance ranking
if not df_feat_rank.empty:
    with open(PASTA_SAIDA/'feature_importance_ranking.pkl','wb') as f: pickle.dump(df_feat_rank,f)

# 12.5 Relatório JSON completo
relatorio = {
    'versao':'V4_AvaliacaoAprofundada',
    'data_execucao':datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'targets_totais':len(TODOS_TARGETS),
    'targets_treinados':len(TARGETS),
    'targets_foco_tcc':len(TARGETS_FOCO),
    'n_treino':int(len(treino)), 'n_teste':int(len(teste)),
    'melhores_modelos':{t:str(a) for t,a in melhores.items()},
    'contagem_vitorias':dict(Counter(melhores.values())),
    'zscore_altman':{
        'formula':"Z''=6.56X1+3.26X2+6.72X3+1.05X4",
        'limiar_segura':ZONA_SEGURA,'limiar_cinza':ZONA_CINZA_INF,
        'obs_validas':int(df_zscore['altman_z_pp'].notna().sum()) if not df_zscore.empty else 0,
        'zonas':df_zscore['zona_altman'].value_counts().to_dict() if not df_zscore.empty else {},
    },
    'score_risco':{
        'n_kpis':len(kpis_disp),'kpis':kpis_disp,
        'classes':dataset['classe_risco'].value_counts().to_dict() if 'classe_risco' in dataset.columns else {},
    },
    'estresse_mc':{'n_sim':N_SIM_MC,'sigma':SIGMA_MC,'combinacoes':len(res_stress)},
    'feature_importance':{
        'n_features':len(df_feat_rank) if not df_feat_rank.empty else 0,
        'top10':df_feat_rank.head(10)['feature'].tolist() if not df_feat_rank.empty else [],
        'familias':df_feat_rank.groupby('familia')['importancia_total'].sum().to_dict() if not df_feat_rank.empty else {},
    },
}
with open(PASTA_SAIDA/'logs'/'relatorio_avaliacao_v4.json','w',encoding='utf-8') as f:
    json.dump(relatorio,f,indent=2,ensure_ascii=False,default=str)

# ── Resumo final ──────────────────────────────────────────────────────────────
print('\n'+'═'*80)
print('RESUMO — 04_cvm_avaliacao.ipynb')
print('═'*80)
print(f'  Targets treinados (Script 3) : {len(TARGETS)} / {len(TODOS_TARGETS)} possíveis')
print(f'  Targets foco TCC             : {len(TARGETS_FOCO)} (Receita, Lucro, EBITDA × 4 horizontes)')
_dom=max(Counter(melhores.values()),key=lambda k:Counter(melhores.values())[k])
print(f'  Algoritmo dominante          : {_dom} ({Counter(melhores.values())[_dom]}× melhor de {len(TARGETS)})')
if not df_zscore.empty and 'altman_z_pp' in df_zscore.columns:
    nv=df_zscore['altman_z_pp'].notna().sum()
    ni=(df_zscore['zona_altman']=='Insolvência').sum()
    print(f'  Z-Score Altman               : {nv:,} obs | {ni:,} em zona de insolvência ({ni/nv*100:.1f}%)')
if not df_stress.empty:
    print(f'  Estresse Monte Carlo         : {len(df_stress)} combinações ({N_SIM_MC} sim. cada)')
print('─'*80)
print('Artefatos em outputs/:')
for a in ['altman_zscore.{csv,parquet}','score_risco_dataset.parquet',
          'analise_estresse_mc.csv','metricas_por_setor.csv',
          'resultados_teste_enriquecido.csv','diagnostico_overfitting.csv',
          'feature_importance_ranking.{csv,pkl}','residuos_detalhados.parquet',
          'melhores_modelos_v4.pkl','zscore_por_empresa.pkl',
          'figuras/{6 figuras .png}','logs/relatorio_avaliacao_v4.json']:
    print(f'  • {a}')
print('═'*80)
print('✅ Script 4 concluído — pronto para 05_cvm_cenarios.ipynb')
logger.info('Script 4 concluído | targets=%d | artefatos persistidos', len(TARGETS))